In [0]:
from pyspark.sql import functions as F
import builtins

DATABASE_NAME = "smart_data_deduplication"

# ---------------------------------------
# 1. Configure your table names
# ---------------------------------------

TARGET_TABLE = f"{DATABASE_NAME}.trusted_records"
QUARANTINE_TABLE = f"{DATABASE_NAME}.quarantine_records"

# ---------------------------------------
# 1a. Perform the incremental MERGE
# (creates trusted_records empty on first
#  run, then upserts clean records by id)
# ---------------------------------------

merge_source_df = spark.table(
    f"{DATABASE_NAME}.clean_records"
)
merge_source_df.createOrReplaceTempView("merge_source")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE}
    AS SELECT * FROM merge_source WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {TARGET_TABLE} AS target
    USING merge_source AS source
    ON target.record_id = source.record_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Use the exact source DataFrame passed to your MERGE operation
# Change merge_source_df if your variable has a different name.
source_count = merge_source_df.count()

# ---------------------------------------
# 2. Get latest MERGE operation metrics
# ---------------------------------------

history_df = spark.sql(
    f"DESCRIBE HISTORY {TARGET_TABLE}"
)

latest_merge = (
    history_df
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc())
    .limit(1)
    .collect()
)

if not latest_merge:
    raise Exception("No MERGE operation found in target table history.")

metrics = latest_merge[0]["operationMetrics"]

inserted = int(metrics.get("numTargetRowsInserted", 0))
updated = int(metrics.get("numTargetRowsUpdated", 0))

# ---------------------------------------
# 3. Count quarantined records
# ---------------------------------------

quarantined = spark.table(QUARANTINE_TABLE).count()

# ---------------------------------------
# 4. Calculate rejected / skipped records
# ---------------------------------------

rejected = max(
    0,
    source_count - inserted - updated - quarantined
)

# ---------------------------------------
# 5. Total processed records
# ---------------------------------------

total_processed = (
    inserted + updated + rejected + quarantined
)

# ---------------------------------------
# 6. Percentage calculation
# ---------------------------------------

def calculate_percentage(value):
    if total_processed == 0:
        return 0.0

    return builtins.round(
        (value / total_processed) * 100, 2
    )

# ---------------------------------------
# 7. Prepare Table 7
# ---------------------------------------

table7_data = [
    ("New records inserted", inserted,
     calculate_percentage(inserted)),

    ("Existing records updated", updated,
     calculate_percentage(updated)),

    ("Records skipped/rejected", rejected,
     calculate_percentage(rejected)),

    ("Records quarantined", quarantined,
     calculate_percentage(quarantined)),

    ("Total processed", total_processed, 100.00)
]

table7_df = spark.createDataFrame(
    table7_data,
    [
        "MERGE Operation",
        "Number of Records",
        "Percentage"
    ]
)

table7_df = table7_df.withColumn(
    "Percentage",
    F.concat(
        F.format_number(F.col("Percentage"), 2),
        F.lit("%")
    )
)

# ---------------------------------------
# 8. Display Table 7
# ---------------------------------------

print("TABLE 7: INCREMENTAL MERGE RESULTS")
print("Source records:", source_count)

display(table7_df)

# ---------------------------------------
# 9. Save results in Delta Lake
# (column names cannot contain spaces)
# ---------------------------------------

table7_df.toDF(
    "merge_operation",
    "number_of_records",
    "percentage"
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{DATABASE_NAME}.incremental_merge_summary"
    )

print("Table 7 saved successfully!")

TABLE 7: INCREMENTAL MERGE RESULTS
Source records: 42


MERGE Operation,Number of Records,Percentage
New records inserted,42,87.50%
Existing records updated,0,0.00%
Records skipped/rejected,0,0.00%
Records quarantined,6,12.50%
Total processed,48,100.00%


Table 7 saved successfully!
